# Phase 11 — Production Prediction / Inference Pipeline
**Problem Statement ID 26184:** Predictive Analytics Framework for Cybercrime Complaints to Forecast Likely Cash Withdrawal Locations

## Objective
Establish a production-ready, leakage-safe inference pipeline that accepts new incoming cybercrime incident complaints and outputs:
1. Binary cashout probability: $P(\text{future\_withdrawal} = 1)$
2. Calibrated 0–100 Risk Score
3. Operational Risk Categories (`LOW`, `MODERATE`, `HIGH`, `CRITICAL`)
4. Human-in-the-loop operational interpretations
5. Local SHAP factor attributions for individual decisions


In [ ]:
import json
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data'
PROC_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = BASE_DIR / 'outputs'
MODELS_DIR = BASE_DIR / 'models'
PRED_DIR = OUTPUTS_DIR / 'predictions'


## 1. Load Model Pipeline
Load the frozen Phase 7 pipeline containing the TRAIN-fitted preprocessor and tuned XGBoost classifier.

In [ ]:
pipeline = joblib.load(MODELS_DIR / 'xgboost_cybercrime_model.pkl')
preprocessor = pipeline.named_steps['preprocessor']
classifier = pipeline.named_steps['classifier']

print(f"Loaded Pipeline: {pipeline}")
print(f"Classifier: {type(classifier)}")


## 2. Load Metadata & Model Version
Inspect model versioning and training metadata.

In [ ]:
with open(MODELS_DIR / 'phase7_xgboost_metadata.json', 'r') as f:
    metadata = json.load(f)

version_report = pd.read_csv(OUTPUTS_DIR / 'phase11_model_version_report.csv')
display(version_report)


## 3. Inspect Expected Model Schema
Verify the 64 features expected by the pipeline.

In [ ]:
artifact_profile = pd.read_csv(OUTPUTS_DIR / 'phase11_model_artifact_profile.csv')
display(artifact_profile)


## 4. Load New Incoming Input Records
Load demonstration synthetic dataset `data/new_prediction_input.csv`.

In [ ]:
new_input_df = pd.read_csv(DATA_DIR / 'new_prediction_input.csv')
print(f"Incoming records: {len(new_input_df)}, Columns: {len(new_input_df.columns)}")
display(new_input_df.head())


## 5. Input Validation & Sanity Checks
Ensure all required features are present, no target-derived leakage exists, and coordinates are valid.

In [ ]:
input_val_report = pd.read_csv(OUTPUTS_DIR / 'phase11_input_validation_report.csv')
display(input_val_report)


## 6. Apply TRAIN-Fitted Preprocessing
Transform raw features into the 556-dimensional space without refitting.

In [ ]:
expected_features = metadata['selected_features']
X_prep = new_input_df[expected_features]
X_trans = preprocessor.transform(X_prep)

print(f"Transformed matrix shape: {X_trans.shape}")
compat_report = pd.read_csv(OUTPUTS_DIR / 'phase11_feature_compatibility_report.csv')
display(compat_report)


## 7. Generate Model Prediction & Probabilities
Generate probability of a qualifying future cash withdrawal: $P(\text{future\_withdrawal} = 1)$.

In [ ]:
probas = pipeline.predict_proba(X_prep)[:, 1]
print(f"Predicted probabilities: {probas}")


## 8. Generate Calibrated Risk Scores
Formula: $\text{risk\_score} = \text{round}(\text{probability} \times 100)$.

In [ ]:
scores = np.round(probas * 100).astype(int)
print(f"Calibrated 0–100 Risk Scores: {scores}")


## 9. Assign Operational Categories & Interpretations
Map scores into operational tiers (`LOW`, `MODERATE`, `HIGH`, `CRITICAL`).

In [ ]:
predictions_df = pd.read_csv(PRED_DIR / 'new_record_predictions.csv')
display(predictions_df)


## 10. Generate Local SHAP Explanations
Explain individual prediction factors using the Phase 10 SHAP explainer integration.

In [ ]:
import shap
clean_names = [n.replace("num__", "").replace("cat__", "") for n in preprocessor.get_feature_names_out()]
explainer = shap.TreeExplainer(classifier)
shap_values = explainer(X_trans)
shap_values.feature_names = clean_names

# Inspect top contributors for first demo sample
idx = 0
row_shap = shap_values.values[idx]
top_pos = np.argsort(row_shap)[::-1][:3]
top_neg = np.argsort(row_shap)[:3]

print(f"Top positive factors for {new_input_df.loc[idx, 'case_id']}:")
for p in top_pos:
    if row_shap[p] > 0:
        print(f"  + {clean_names[p]}: {row_shap[p]:+.4f}")

print(f"Top negative factors for {new_input_df.loc[idx, 'case_id']}:")
for n in top_neg:
    if row_shap[n] < 0:
        print(f"  - {clean_names[n]}: {row_shap[n]:+.4f}")


## 11. Batch Prediction Logging
Inspect persistent prediction log entries.

In [ ]:
pred_log = pd.read_csv(PRED_DIR / 'prediction_log.csv')
print(f"Total logged predictions: {len(pred_log)}")
display(pred_log.tail(5))


## 12. Prediction Integrity Validation
Verify mathematical identity between probability and risk score, score ranges, and category alignment.

In [ ]:
pred_val = pd.read_csv(OUTPUTS_DIR / 'phase11_prediction_validation.csv')
display(pred_val)

drift_report = pd.read_csv(OUTPUTS_DIR / 'phase11_input_drift_report.csv')
display(drift_report)


## 13. Leakage & Sensitive Data Audits
Verify zero leakage and no raw financial credentials exposed.

In [ ]:
leakage_report = pd.read_csv(OUTPUTS_DIR / 'phase11_leakage_audit.csv')
display(leakage_report)

sens_report = pd.read_csv(OUTPUTS_DIR / 'phase11_sensitive_data_audit.csv')
display(sens_report)


## 14. Conclusions & Next Steps
1. **Reusable Inference Engine:** `src/predict.py` operates as an end-to-end inference script supporting single records and arbitrary CSV batches.
2. **Strict Guardrails:** Validates against target leakage, PII exposure, coordinate corruption, and feature ordering mismatches.
3. **Interpretability:** Connects seamless probability generation to calibrated 0–100 scores, operational triage categories, and local SHAP attributions.

**Phase 11 is COMPLETE. Ready for Phase 12 — Spatial Hotspot & Geographic Cluster Analysis.**
